# Phase 9 — Nonequilibrium Thermodynamic Interpretation and Entropy-Production-Related Analysis

## Synthetic equilibrium vs nonequilibrium benchmark

Phase 9 begins with a stochastic system whose thermodynamic behavior is analytically known before any estimator is applied to the experimental MSC01 trajectories.

The benchmark is the two-dimensional rotational Ornstein–Uhlenbeck process

$$
dX = A X\,dt + \sqrt{2D}\,dW,
$$

with

$$
A = -kI + \omega R.
$$

The fixed baseline parameters are:

- \(k = 1\)
- \(D = 1\)
- \(dt = 0.1\)
- \(n_{\mathrm{steps}} = 100000\)
- seed \(= 2031\)

Two conditions are compared:

- equilibrium: \(\omega = 0\)
- nonequilibrium: \(\omega = 1\)

For both conditions, the stationary covariance is

$$
C = \frac{D}{k}I = I.
$$

Therefore, the two systems can have the same stationary spatial distribution even though their dynamics differ.

For the rotational nonequilibrium process, the analytical physical entropy-production rate is

$$
\sigma = \frac{2\omega^2}{k}.
$$

Thus the pre-specified analytical values are:

- equilibrium: \(\sigma = 0\)
- nonequilibrium: \(\sigma = 2\)

Before estimating entropy production from paths, this section tests two simpler properties:

1. whether both simulations reproduce the expected stationary covariance;
2. whether the nonequilibrium process displays the expected signed rotational probability current.

The rotational-current observable is

$$
c_t =
x_t y_{t+1}
-
y_t x_{t+1}.
$$

Its stationary expectation is

$$
E[c_t]
=
\frac{2D}{k}
e^{-k\,dt}
\sin(\omega\,dt).
$$

This rotational-current observable is **not** an entropy-production estimator.

In [1]:
import numpy as np
import pandas as pd

from _path import PROJECT_ROOT

from src.thermo import (
    analytic_epr_rotational_ou,
    analytic_mean_rotational_increment,
    rotational_increments,
    simulate_rotational_ou,
    stationary_covariance_isotropic,
)

print("Project root:", PROJECT_ROOT)
print("NumPy version:", np.__version__)
print("Phase 9 synthetic setup: READY")

Project root: C:\Users\Abolfazl.PH\Desktop\cell-irreversibility
NumPy version: 2.0.1
Phase 9 synthetic setup: READY


In [2]:
K = 1.0
DIFFUSION = 1.0
DT = 0.1
N_STEPS = 100_000
SYNTHETIC_SEED = 2031

OMEGA_EQUILIBRIUM = 0.0
OMEGA_NONEQUILIBRIUM = 1.0


stationary_covariance_theory = (
    stationary_covariance_isotropic(
        k=K,
        diffusion=DIFFUSION,
    )
)

sigma_equilibrium_theory = (
    analytic_epr_rotational_ou(
        k=K,
        omega=OMEGA_EQUILIBRIUM,
    )
)

sigma_nonequilibrium_theory = (
    analytic_epr_rotational_ou(
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
    )
)

current_equilibrium_theory = (
    analytic_mean_rotational_increment(
        k=K,
        omega=OMEGA_EQUILIBRIUM,
        diffusion=DIFFUSION,
        dt=DT,
    )
)

current_nonequilibrium_theory = (
    analytic_mean_rotational_increment(
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
        diffusion=DIFFUSION,
        dt=DT,
    )
)


print("Fixed synthetic parameters")
print("--------------------------")
print("k:", K)
print("D:", DIFFUSION)
print("dt:", DT)
print("n_steps:", N_STEPS)
print("seed:", SYNTHETIC_SEED)

print()
print("Theoretical stationary covariance:")
print(stationary_covariance_theory)

print()
print("Theoretical physical EPR")
print("equilibrium:", sigma_equilibrium_theory)
print(
    "nonequilibrium:",
    sigma_nonequilibrium_theory,
)

print()
print("Theoretical mean rotational increment")
print(
    "equilibrium:",
    current_equilibrium_theory,
)
print(
    "nonequilibrium:",
    current_nonequilibrium_theory,
)

Fixed synthetic parameters
--------------------------
k: 1.0
D: 1.0
dt: 0.1
n_steps: 100000
seed: 2031

Theoretical stationary covariance:
[[1. 0.]
 [0. 1.]]

Theoretical physical EPR
equilibrium: 0.0
nonequilibrium: 2.0

Theoretical mean rotational increment
equilibrium: 0.0
nonequilibrium: 0.18066602190484835


In [3]:
equilibrium_path = simulate_rotational_ou(
    n_steps=N_STEPS,
    k=K,
    omega=OMEGA_EQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
    seed=SYNTHETIC_SEED,
)

nonequilibrium_path = simulate_rotational_ou(
    n_steps=N_STEPS,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
    seed=SYNTHETIC_SEED,
)


print(
    "Equilibrium path shape:",
    equilibrium_path.shape,
)

print(
    "Nonequilibrium path shape:",
    nonequilibrium_path.shape,
)

print(
    "All equilibrium values finite:",
    np.isfinite(equilibrium_path).all(),
)

print(
    "All nonequilibrium values finite:",
    np.isfinite(nonequilibrium_path).all(),
)

Equilibrium path shape: (100001, 2)
Nonequilibrium path shape: (100001, 2)
All equilibrium values finite: True
All nonequilibrium values finite: True


In [4]:
equilibrium_covariance_empirical = np.cov(
    equilibrium_path.T,
    ddof=1,
)

nonequilibrium_covariance_empirical = np.cov(
    nonequilibrium_path.T,
    ddof=1,
)


equilibrium_rotational = rotational_increments(
    equilibrium_path
)

nonequilibrium_rotational = rotational_increments(
    nonequilibrium_path
)


equilibrium_current_empirical = (
    equilibrium_rotational.mean()
)

nonequilibrium_current_empirical = (
    nonequilibrium_rotational.mean()
)


synthetic_summary = pd.DataFrame(
    {
        "condition": [
            "equilibrium",
            "nonequilibrium",
        ],
        "omega": [
            OMEGA_EQUILIBRIUM,
            OMEGA_NONEQUILIBRIUM,
        ],
        "theory_epr": [
            sigma_equilibrium_theory,
            sigma_nonequilibrium_theory,
        ],
        "theory_mean_rotational_increment": [
            current_equilibrium_theory,
            current_nonequilibrium_theory,
        ],
        "empirical_mean_rotational_increment": [
            equilibrium_current_empirical,
            nonequilibrium_current_empirical,
        ],
        "empirical_var_x": [
            equilibrium_covariance_empirical[0, 0],
            nonequilibrium_covariance_empirical[0, 0],
        ],
        "empirical_var_y": [
            equilibrium_covariance_empirical[1, 1],
            nonequilibrium_covariance_empirical[1, 1],
        ],
        "empirical_cov_xy": [
            equilibrium_covariance_empirical[0, 1],
            nonequilibrium_covariance_empirical[0, 1],
        ],
    }
)

synthetic_summary

,condition,omega,theory_epr,theory_mean_rotational_increment,empirical_mean_rotational_increment,empirical_var_x,empirical_var_y,empirical_cov_xy
0,equilibrium,0.0,0.0,0.000000,-0.00034,0.992851,0.983713,-0.000967
1,nonequilibrium,1.0,2.0,0.180666,0.17810,0.992347,0.984064,-0.001169


### Initial synthetic benchmark result

The exact-transition simulations reproduce the expected stationary spatial statistics in both conditions.

For the equilibrium process, the empirical covariance is close to the theoretical stationary covariance

$$
C = I,
$$

and the empirical mean signed rotational increment is approximately zero:

$$
\langle c_t\rangle_{\mathrm{emp}}
=
-0.00034.
$$

For the nonequilibrium rotational process, the empirical covariance remains close to the same stationary covariance, but the dynamics display a clear positive rotational current.

The analytical prediction is

$$
\langle c_t\rangle_{\mathrm{theory}}
=
0.180666,
$$

while the simulation gives

$$
\langle c_t\rangle_{\mathrm{emp}}
=
0.17810.
$$

Thus, the equilibrium and nonequilibrium systems can have nearly indistinguishable stationary spatial distributions while exhibiting different temporal probability currents.

This demonstrates why stationary density alone is insufficient to diagnose nonequilibrium dynamics.

The rotational-current observable used here is a dynamical diagnostic and is **not** itself an entropy-production estimate.

## Path-probability-ratio validation criterion

The next benchmark directly compares forward and time-reversed path probabilities for the exactly sampled rotational Ornstein–Uhlenbeck process.

Three rates will be kept distinct:

$$
\sigma_{\mathrm{continuous}}
=
\frac{2\omega^2}{k},
$$

the continuous-time physical entropy-production rate;

$$
\dot I_{\mathrm{sampled,theory}}
=
\frac{
4e^{-2k\,dt}\sin^2(\omega dt)
}{
(1-e^{-2k\,dt})dt
},
$$

the exact forward/reverse path-space irreversibility rate of the process observed only every \(dt\);

and

$$
\dot I_{\mathrm{path,empirical}}
=
\frac{
\log P[\Gamma]
-
\log P[\Gamma^R]
}{
N_{\mathrm{steps}}\,dt
},
$$

the empirical rate calculated from the simulated trajectory.

The validation criteria are fixed before inspecting the empirical path-ratio result.

For the nonequilibrium benchmark:

- the theoretical sampled path-space rate must be positive;
- it must be smaller than the continuous-time physical entropy-production rate at the finite sampling interval \(dt=0.1\);
- the empirical path-log-ratio rate must agree with the exact sampled-theory rate to within **5% relative error**.

For the equilibrium benchmark:

- the theoretical continuous and sampled rates are exactly zero;
- because detailed balance holds analytically, the empirical path-log-ratio rate should be zero up to floating-point numerical error;
- an absolute empirical rate not exceeding \(10^{-10}\) simulation-time\(^{-1}\) will be treated as numerically zero.

These criteria are numerical-validation criteria for the synthetic benchmark. They are not significance thresholds for the later MSC01 experimental analysis.

The 5% tolerance will not be changed after inspection of the empirical path-ratio result merely to obtain a passing validation.

In [5]:
from src.thermo import (
    analytic_sampled_path_irreversibility_rate,
    ou_path_log_ratio,
)


total_time = N_STEPS * DT


equilibrium_path_log_ratio = ou_path_log_ratio(
    path=equilibrium_path,
    k=K,
    omega=OMEGA_EQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
)

nonequilibrium_path_log_ratio = ou_path_log_ratio(
    path=nonequilibrium_path,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
)


equilibrium_empirical_rate = (
    equilibrium_path_log_ratio
    / total_time
)

nonequilibrium_empirical_rate = (
    nonequilibrium_path_log_ratio
    / total_time
)


equilibrium_sampled_theory = (
    analytic_sampled_path_irreversibility_rate(
        k=K,
        omega=OMEGA_EQUILIBRIUM,
        dt=DT,
    )
)

nonequilibrium_sampled_theory = (
    analytic_sampled_path_irreversibility_rate(
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
        dt=DT,
    )
)


print("Total simulated time:", total_time)

print()
print("Equilibrium")
print("-----------")
print(
    "continuous physical EPR:",
    sigma_equilibrium_theory,
)
print(
    "sampled path-space theory:",
    equilibrium_sampled_theory,
)
print(
    "empirical path log ratio:",
    equilibrium_path_log_ratio,
)
print(
    "empirical path-space rate:",
    equilibrium_empirical_rate,
)

print()
print("Nonequilibrium")
print("----------------")
print(
    "continuous physical EPR:",
    sigma_nonequilibrium_theory,
)
print(
    "sampled path-space theory:",
    nonequilibrium_sampled_theory,
)
print(
    "empirical path log ratio:",
    nonequilibrium_path_log_ratio,
)
print(
    "empirical path-space rate:",
    nonequilibrium_empirical_rate,
)

Total simulated time: 10000.0

Equilibrium
-----------
continuous physical EPR: 0.0
sampled path-space theory: 0.0
empirical path log ratio: -1.4551915228366852e-11
empirical path-space rate: -1.4551915228366853e-15

Nonequilibrium
----------------
continuous physical EPR: 2.0
sampled path-space theory: 1.8006480429063036
empirical path log ratio: 17750.719849065208
empirical path-space rate: 1.7750719849065208


In [6]:
nonequilibrium_relative_error = (
    abs(
        nonequilibrium_empirical_rate
        - nonequilibrium_sampled_theory
    )
    / nonequilibrium_sampled_theory
)

equilibrium_absolute_rate = abs(
    equilibrium_empirical_rate
)


path_ratio_validation = pd.DataFrame(
    {
        "condition": [
            "equilibrium",
            "nonequilibrium",
        ],
        "continuous_physical_epr": [
            sigma_equilibrium_theory,
            sigma_nonequilibrium_theory,
        ],
        "sampled_path_theory_rate": [
            equilibrium_sampled_theory,
            nonequilibrium_sampled_theory,
        ],
        "empirical_path_rate": [
            equilibrium_empirical_rate,
            nonequilibrium_empirical_rate,
        ],
    }
)


equilibrium_pass = (
    equilibrium_absolute_rate
    <= 1e-10
)

nonequilibrium_ordering_pass = (
    0.0
    < nonequilibrium_sampled_theory
    < sigma_nonequilibrium_theory
)

nonequilibrium_accuracy_pass = (
    nonequilibrium_relative_error
    <= 0.05
)


print(path_ratio_validation.to_string(index=False))

print()
print(
    "Nonequilibrium relative error:",
    nonequilibrium_relative_error,
)

print()
print("Pre-specified validation checks")
print("--------------------------------")
print(
    "Equilibrium numerical-zero criterion:",
    equilibrium_pass,
)
print(
    "Finite-sampling ordering criterion:",
    nonequilibrium_ordering_pass,
)
print(
    "Nonequilibrium <= 5% relative-error criterion:",
    nonequilibrium_accuracy_pass,
)

print()
print(
    "Overall path-ratio validation:",
    (
        equilibrium_pass
        and nonequilibrium_ordering_pass
        and nonequilibrium_accuracy_pass
    ),
)

     condition  continuous_physical_epr  sampled_path_theory_rate  empirical_path_rate
   equilibrium                      0.0                  0.000000        -1.455192e-15
nonequilibrium                      2.0                  1.800648         1.775072e+00

Nonequilibrium relative error: 0.014203807401751993

Pre-specified validation checks
--------------------------------
Equilibrium numerical-zero criterion: True
Finite-sampling ordering criterion: True
Nonequilibrium <= 5% relative-error criterion: True

Overall path-ratio validation: True


### Path-probability-ratio validation result

The pre-specified synthetic path-ratio validation passed.

For the equilibrium process, the empirical path-space irreversibility rate was

$$
-1.46\times10^{-15},
$$

which is numerically zero and comfortably satisfies the pre-specified absolute tolerance of

$$
10^{-10}.
$$

This is consistent with detailed balance: forward and time-reversed paths have equal probability in the equilibrium benchmark.

For the nonequilibrium rotational process, the continuous-time physical entropy-production rate is

$$
\sigma_{\mathrm{continuous}} = 2.000000.
$$

At the finite observation interval

$$
dt = 0.1,
$$

the exact sampled path-space irreversibility rate is

$$
\dot I_{\mathrm{sampled,theory}}
=
1.800648.
$$

The empirical long-trajectory path-log-ratio calculation gave

$$
\dot I_{\mathrm{path,empirical}}
=
1.775072.
$$

The relative error between the empirical rate and the exact sampled-theory prediction was

$$
0.014204
\approx
1.42\%.
$$

This is below the pre-specified 5% validation tolerance.

All pre-specified validation checks passed.

The difference between the continuous-time physical entropy-production rate and the sampled path-space rate is not a simulation error. It reflects loss of observable temporal information caused by finite-time sampling.

Thus, even with exact stochastic dynamics and exact transition likelihoods, temporal coarse-graining can reduce the irreversibility observable from sampled trajectories.

## Spatial coarse-graining and hidden dissipation

The full two-dimensional rotational Ornstein–Uhlenbeck process is nonequilibrium.

For the pre-specified benchmark,

$$
k=1,\qquad
D=1,\qquad
\omega=1,
$$

the continuous-time physical entropy-production rate is

$$
\sigma_{\mathrm{continuous}}=2.
$$

At the finite observation interval

$$
dt=0.1,
$$

the exact full two-dimensional sampled path-space irreversibility rate is

$$
\dot I_{\mathrm{full,sampled}}
=
1.800648.
$$

However, suppose that only one Cartesian coordinate is observed and the other coordinate is hidden.

For either observed coordinate, the stationary autocovariance is

$$
C_x(\tau)
=
\frac{D}{k}
e^{-k|\tau|}
\cos(\omega\tau).
$$

Because this covariance is invariant under time reversal, the stationary one-coordinate Gaussian process has the same probability for a temporal block and its reversed block.

Therefore,

$$
D_{KL}
\left(
P[x_0,\ldots,x_n]
\parallel
P[x_n,\ldots,x_0]
\right)
=
0.
$$

The same statement holds if only the \(y\) coordinate is observed.

Thus, this benchmark provides an explicit example in which

$$
\text{physical entropy production} > 0
$$

while

$$
\text{observable one-coordinate path irreversibility} = 0.
$$

This is an example of **hidden dissipation under coarse-graining**.

Importantly, the reduced one-coordinate process is not assumed to be first-order Markov. Its complete finite-block Gaussian distribution is used for the numerical check below.

In [7]:
from src.thermo import projected_scalar_path_log_ratio


COARSE_GRAIN_CHECK_STATES = 25

scalar_x_segment = (
    nonequilibrium_path[
        :COARSE_GRAIN_CHECK_STATES,
        0,
    ]
)

scalar_y_segment = (
    nonequilibrium_path[
        :COARSE_GRAIN_CHECK_STATES,
        1,
    ]
)


scalar_x_log_ratio = projected_scalar_path_log_ratio(
    values=scalar_x_segment,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
)

scalar_y_log_ratio = projected_scalar_path_log_ratio(
    values=scalar_y_segment,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
)


segment_duration = (
    (COARSE_GRAIN_CHECK_STATES - 1)
    * DT
)


scalar_x_rate = (
    scalar_x_log_ratio
    / segment_duration
)

scalar_y_rate = (
    scalar_y_log_ratio
    / segment_duration
)


print(
    "States in scalar check:",
    COARSE_GRAIN_CHECK_STATES,
)

print(
    "Segment duration:",
    segment_duration,
)

print()
print(
    "x-only path log ratio:",
    scalar_x_log_ratio,
)

print(
    "x-only path-space rate:",
    scalar_x_rate,
)

print()
print(
    "y-only path log ratio:",
    scalar_y_log_ratio,
)

print(
    "y-only path-space rate:",
    scalar_y_rate,
)

States in scalar check: 25
Segment duration: 2.4000000000000004

x-only path log ratio: -1.4210854715202004e-14
x-only path-space rate: -5.921189464667501e-15

y-only path log ratio: 1.7763568394002505e-15
y-only path-space rate: 7.401486830834376e-16


In [8]:
coarse_graining_summary = pd.DataFrame(
    {
        "observation": [
            "continuous full system",
            "sampled full 2D",
            "x coordinate only",
            "y coordinate only",
        ],
        "rate": [
            sigma_nonequilibrium_theory,
            nonequilibrium_empirical_rate,
            scalar_x_rate,
            scalar_y_rate,
        ],
        "interpretation": [
            "physical entropy-production rate",
            "empirical sampled path-space irreversibility rate",
            "coarse-grained observed path-space rate",
            "coarse-grained observed path-space rate",
        ],
    }
)

coarse_graining_summary

,observation,rate,interpretation
0,continuous full system,2.000000e+00,physical entropy-production rate
1,sampled full 2D,1.775072e+00,empirical sampled path-space irreversibility rate
2,x coordinate only,-5.921189e-15,coarse-grained observed path-space rate
3,y coordinate only,7.401487e-16,coarse-grained observed path-space rate


### Spatial coarse-graining result

The spatial coarse-graining benchmark produced the analytically expected result.

The complete nonequilibrium system has the continuous-time physical entropy-production rate

$$
\sigma_{\mathrm{continuous}} = 2.
$$

For the full two-dimensional process sampled every

$$
dt = 0.1,
$$

the exact theoretical sampled path-space irreversibility rate is

$$
\dot I_{\mathrm{full,sampled,theory}}
=
1.800648,
$$

and the long simulated trajectory gave

$$
\dot I_{\mathrm{full,sampled,empirical}}
=
1.775072.
$$

However, when either Cartesian coordinate is observed alone, the numerical path-log-ratio rates are

$$
\dot I_x
=
-5.92\times10^{-15}
$$

and

$$
\dot I_y
=
7.40\times10^{-16}.
$$

These values are numerically zero and are consistent with the exact analytical result that the stationary one-coordinate Gaussian process is invariant under time reversal.

Therefore, this synthetic system demonstrates that

$$
\text{physical entropy production} > 0
$$

can coexist with

$$
\text{observable irreversibility in a coarse-grained variable} = 0.
$$

The disappearance of the time-direction signal after hiding one coordinate is an example of **hidden dissipation under coarse-graining**.

This result is especially important for the later MSC01 interpretation: a weak or null trajectory-level irreversibility signal cannot be interpreted as evidence for zero microscopic cellular dissipation.